# Task 1: Agent Concepts & Mental Model

## Agent vs. chatbot vs. workflow

- **Chatbot:** A system that responds to a user's message, usually one turn at a time. It may remember conversation context, but it does not necessarily take actions or pursue a goal independently.
- **Workflow:** A predefined sequence of steps with fixed rules and branching. The developer decides the order of operations, so the same kind of input follows roughly the same process every time.
- **Agent:** A system that uses an LLM to pursue a goal by deciding what to do next. It can choose tools, use their results, keep track of intermediate state, and continue until it reaches a useful result or needs help.

Something becomes **agentic** when it has meaningful autonomy rather than only generating a reply. Typical signs are:

1. **Autonomy:** It chooses the next step instead of following only a hard-coded sequence.
2. **Tool use:** It can call APIs, search data, run code, or interact with external systems.
3. **Multi-step planning:** It breaks a larger goal into smaller actions and carries context between them.
4. **Self-correction:** It evaluates observations or errors, revises its approach, and tries again when appropriate.

## ReAct pattern

ReAct means **Reason -> Act -> Observe -> repeat**. The model reasons about the current goal, takes an action such as calling a tool, observes the result, and then uses that result to decide whether another action is needed.

```text
+-------+     +-----+     +---------+
| Reason| --> | Act | --> | Observe |
+-------+     +-----+     +---------+
    ^                            |
    |                            v
    +-------- repeat --------- Goal reached?
                                  |
                              Yes: answer
```

Pseudocode:

```python
while not goal_is_complete:
    thought = model.reason(goal, history)

    if thought.requires_tool:
        observation = call_tool(thought.tool, thought.arguments)
        history.append((thought, observation))
    else:
        return thought.final_answer
```

The loop is the important idea: the model is not just producing one response. It is using feedback from the environment to choose the next step.

## When an agent is overkill

An agent is overkill when a single prompt can answer the question or when a short deterministic script can complete the task reliably. Agents add latency, API cost, complexity, and more ways to fail, so a fixed workflow is usually better for predictable operations such as formatting data or validating a known schema. Use an agent when the path cannot be fully specified in advance and the ability to choose tools or recover from intermediate results provides real value.

# Task 2: Tool Calling Fundamentals

This notebook uses the Gemini API because an Anthropic API key is not available. The provider-specific names are different, but the tool-calling flow is the same:

- **`name`** identifies the function the model can call.
- **`description`** explains what the tool does and when it should be used.
- **`parameters`** is Gemini's JSON Schema equivalent of Anthropic's **`input_schema`**. It defines the argument types, required fields, and allowed values.

Tool descriptions matter because the model uses them to select the right tool and construct valid arguments. A specific description reduces ambiguous calls, while a precise schema helps the API validate the argument shape and tells the model which fields are required, what types they have, and what values are allowed.

The example below defines a calculator and a weather lookup stub. The first API request asks the model to answer a weather question and gives it permission to use the tools. The Python code then manually executes whichever tool the model selects and packages the result with Gemini's `Part.from_function_response`, which is the provider-equivalent of Anthropic's `tool_result` block before making the follow-up request.

In [8]:
import os
from getpass import getpass
from google import genai
from google.genai import types


function_declarations = [
    {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression using numbers and +, -, *, /, and parentheses.",
        "parameters": {
            "type": "OBJECT",
            "properties": {
                "expression": {
                    "type": "STRING",
                    "description": "The arithmetic expression to evaluate, such as '(12 + 8) / 4'.",
                }
            },
            "required": ["expression"],
        },
    },
    {
        "name": "weather_lookup",
        "description": "Return stub weather data for a city. Use this when the user asks about current weather.",
        "parameters": {
            "type": "OBJECT",
            "properties": {
                "city": {
                    "type": "STRING",
                    "description": "The city whose weather should be looked up, such as 'London'.",
                },
                "unit": {
                    "type": "STRING",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "The temperature unit for the result.",
                },
            },
            "required": ["city"],
        },
    },
]


def execute_tool(name, arguments):
    """Manually dispatch the tool selected by the model."""
    if name == "calculator":
        allowed_names = {"__builtins__": {}}
        result = eval(arguments["expression"], allowed_names, {})
        return {"expression": arguments["expression"], "result": result}

    if name == "weather_lookup":
        unit = arguments.get("unit", "celsius")
        temperatures = {
            "london": 18,
            "tokyo": 26,
            "new york": 22,
        }
        base_temperature = temperatures.get(arguments["city"].lower(), 20)
        temperature = base_temperature if unit == "celsius" else round(base_temperature * 9 / 5 + 32, 1)
        return {
            "city": arguments["city"],
            "temperature": temperature,
            "unit": unit,
            "conditions": "Partly cloudy",
            "source": "stub data for learning",
        }

    raise ValueError(f"Unknown tool: {name}")


api_key = os.environ.get("GEMINI_API_KEY") or getpass("Enter your Gemini API key: ")
client = genai.Client(api_key=api_key)
config = types.GenerateContentConfig(
    tools=[types.Tool(function_declarations=function_declarations)],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)
model = "gemini-3.6-flash"
prompt = "What is the current weather in London? Give the temperature in celsius."

# Request 1: the model decides whether a tool is needed.
chat = client.chats.create(model=model, config=config)
response = chat.send_message(prompt)
function_call = response.function_calls[0]
tool_output = execute_tool(function_call.name, function_call.args)

# Request 2: send the observed tool result so Gemini can write the final answer.
tool_result = types.Part.from_function_response(
    name=function_call.name,
    response=tool_output,
)
final_response = chat.send_message(tool_result)

print("Tool selected:", function_call.name)
print("Tool arguments:", function_call.args)
print("tool_result:", tool_output)
print("Final answer:", final_response.text)

Tool selected: weather_lookup
Tool arguments: {'city': 'London', 'unit': 'celsius'}
tool_result: {'city': 'London', 'temperature': 18, 'unit': 'celsius', 'conditions': 'Partly cloudy', 'source': 'stub data for learning'}
Final answer: The current weather in London is partly cloudy with a temperature of 18°C.


# Task 3: Build a Minimal Agent Loop

The agent loop repeatedly sends the conversation to Gemini, checks whether the response contains function calls, executes each selected tool locally, and sends the function responses back to the model. It stops when Gemini returns text without a function call or when `max_iterations` is reached.

A maximum iteration count is an important safety guard: it prevents a faulty model response, tool error, or unexpected prompt from creating an infinite API loop.

In [10]:
def run_agent(user_prompt, max_iterations=6):
    """Run a bounded Gemini tool-calling loop and return the final text answer."""
    chat = client.chats.create(model=model, config=config)
    working_memory = {
        "iteration": 0,
        "tool_calls": [],
        "observations": [],
    }

    print("[agent] Step 1: sending the user message")
    response = chat.send_message(user_prompt)

    for iteration in range(1, max_iterations + 1):
        working_memory["iteration"] = iteration
        function_calls = response.function_calls

        if not function_calls:
            print(f"[agent] Step {iteration}: final text answer received")
            return {
                "answer": response.text,
                "iterations": iteration,
                "tool_calls": working_memory["tool_calls"],
                "working_memory": working_memory,
            }

        print(
            f"[agent] Step {iteration}: model requested "
            f"{len(function_calls)} tool call(s)"
        )
        tool_results = []
        for function_call in function_calls:
            print(
                f"[tool call] {function_call.name} "
                f"arguments={dict(function_call.args)}"
            )
            tool_output = execute_tool(function_call.name, function_call.args)
            working_memory["tool_calls"].append(
                {
                    "name": function_call.name,
                    "arguments": dict(function_call.args),
                    "result": tool_output,
                }
            )
            working_memory["observations"].append(tool_output)
            print(f"[observation] {tool_output}")
            tool_results.append(
                types.Part.from_function_response(
                    name=function_call.name,
                    response=tool_output,
                )
            )

        print(f"[agent] Step {iteration}: sending tool observations back to Gemini")
        response = chat.send_message(tool_results)

    raise RuntimeError(
        f"Agent stopped after reaching max_iterations={max_iterations} "
        "without a final text answer."
    )


multi_step_prompt = (
    "Look up the weather in London and Tokyo in celsius, then tell me "
    "which city is warmer and by how many degrees."
)
agent_result = run_agent(multi_step_prompt, max_iterations=6)

print("Iterations:", agent_result["iterations"])
print("Tool calls:")
for tool_call in agent_result["tool_calls"]:
    print(tool_call)
print("Final answer:", agent_result["answer"])

[agent] Step 1: sending the user message
[agent] Step 1: model requested 2 tool call(s)
[tool call] weather_lookup arguments={'unit': 'celsius', 'city': 'London'}
[observation] {'city': 'London', 'temperature': 18, 'unit': 'celsius', 'conditions': 'Partly cloudy', 'source': 'stub data for learning'}
[tool call] weather_lookup arguments={'unit': 'celsius', 'city': 'Tokyo'}
[observation] {'city': 'Tokyo', 'temperature': 26, 'unit': 'celsius', 'conditions': 'Partly cloudy', 'source': 'stub data for learning'}
[agent] Step 1: sending tool observations back to Gemini
[agent] Step 2: model requested 1 tool call(s)
[tool call] calculator arguments={'expression': '26 - 18'}
[observation] {'expression': '26 - 18', 'result': 8}
[agent] Step 2: sending tool observations back to Gemini
[agent] Step 3: final text answer received
Iterations: 3
Tool calls:
{'name': 'weather_lookup', 'arguments': {'unit': 'celsius', 'city': 'London'}, 'result': {'city': 'London', 'temperature': 18, 'unit': 'celsius', 

# Task 4: Memory & State Handling

## Conversation memory vs. working memory

- **Conversation memory** is the message history sent to the model. In this notebook, the Gemini `chat` object retains the user prompt, model function calls, and function-response messages so the model can use earlier context on the next turn.
- **Working memory** is the agent's temporary program state while it completes the task. The `working_memory` dictionary tracks the current iteration, tool calls, and observations so Python can control the loop, log progress, and inspect what happened.

Conversation memory helps the model understand the dialogue. Working memory helps the program manage execution. They overlap in content, but they have different owners: Gemini uses the conversation history, while Python owns the working state.

# Task 5: Failure Modes & Guardrails

Agents fail at the boundary between model output, tool execution, and loop control. The following demonstration intentionally sends invalid tool requests to the local dispatcher and catches the errors so the notebook can document the failure without stopping execution.

## Failure modes and mitigations

| Failure mode | What can happen | Mitigation |
| --- | --- | --- |
| Infinite loop | The model keeps requesting tools and never produces a final answer. | Enforce `max_iterations` and raise a visible error when the limit is reached. |
| Hallucinated tool call | The model selects a tool name that is not registered. | Keep an allowlist of registered tools and reject unknown names explicitly. |
| Wrong tool arguments | Required fields are missing or have the wrong type. | Validate arguments against the tool's JSON Schema before execution. |
| Silent tool errors | A tool fails, but the agent continues as if it received valid data. | Catch exceptions, record structured error results, and show the error to the model and user. |
| Unsafe tool input | A calculator, file reader, or API tool receives malicious or dangerous input. | Restrict permitted operations, apply timeouts, and isolate or sandbox side effects. |
| Provider/API failure | Rate limits, authentication errors, or temporary server failures interrupt a request. | Add bounded retries with backoff and return a clear failure state after retry exhaustion. |

Frameworks such as LangChain, LangGraph, and CrewAI exist because production agents need reusable implementations of these concerns: message and state management, tool registries, retries, branching, persistence, tracing, and human approval. Building the loop by hand makes the core idea clear, while a framework reduces the amount of reliability and orchestration code that must be maintained as the system grows.

In [11]:
def demonstrate_failure(label, tool_name, arguments):
    print(f"\n--- {label} ---")
    print(f"Requested tool: {tool_name}")
    print(f"Arguments: {arguments}")
    try:
        result = execute_tool(tool_name, arguments)
    except Exception as error:
        print(f"Observed failure: {type(error).__name__}: {error}")
        return
    print(f"Unexpected success: {result}")


demonstrate_failure(
    "Undefined tool",
    "currency_lookup",
    {"from_currency": "USD", "to_currency": "EUR", "amount": 10},
)
demonstrate_failure(
    "Missing required argument",
    "weather_lookup",
    {},
)


--- Undefined tool ---
Requested tool: currency_lookup
Arguments: {'from_currency': 'USD', 'to_currency': 'EUR', 'amount': 10}
Observed failure: ValueError: Unknown tool: currency_lookup

--- Missing required argument ---
Requested tool: weather_lookup
Arguments: {}
Observed failure: KeyError: 'city'
